# 5. Classifier-Free Guidance

So far the denoiser generates images at random. This notebook adds **steering**: given a class label (airplane, dog, ship…) the sampler can be guided toward images that look like that class — and the guidance can be amplified beyond anything in the training data.

By the end you will have:
- Extended the MLP denoiser to accept a class label
- Fine-tuned the notebook 03 checkpoint with conditional training
- Implemented the CFG noise estimate
- Generated class-conditional CIFAR-10 images and swept the guidance scale

In [2]:
# @title Setup — run this cell first (Colab only)
# !git clone https://github.com/maigimenez/let-it-rip
# %cd let-it-rip
# !pip install -q "jax[cuda12]" plotly jaxtyping optax datasets pillow

In [3]:
import os
import pickle

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import optax
import plotly.graph_objects as go
from datasets import load_dataset
from jaxtyping import Array, Float, Int, PRNGKeyArray
from plotly.subplots import make_subplots

CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]
NULL_CLASS = 10  # index reserved for the unconditional (null) embedding

print(f"JAX {jax.__version__} · backend: {jax.default_backend()}")

JAX 0.10.1 · backend: cpu


---
## 1. Why CFG?

The key question: how do you get a single model to steer generation toward a class *and* generate freely without one?

- **Conditional-only model** $ε_θ(x_t, t, c)$: learns to denoise toward class `c`, but inference is bounded by the training distribution. You can't adjust the class signal beyond what the model saw during training.

- **Conditional Free Guidance** (CFG) trains one model for both tasks. During training, the class label is randomly replaced with a null token `∅` with probability `p_uncond ≈ 0.1`, so the model learns the conditional distribution $ε_θ(x_t, t, c)$ *and* the unconditional distribution $ε_θ(x_t, t, ∅)$ in one pass. At inference, you combine them:

$$\hat\varepsilon_\text{CFG} = \underbrace{\varepsilon_\theta(x_t, t, \varnothing)}_{\text{unconditional}} + w\,\bigl(\underbrace{\varepsilon_\theta(x_t, t, c)}_{\text{conditional}} - \varepsilon_\theta(x_t, t, \varnothing)\bigr)$$

The guidance scale `w` extrapolates beyond the training distribution in the direction of the class:
- `w = 1` → plain conditional model, no amplification
- `w > 1` → pushes toward "more class-like than any training example"
- `w >> 1` → sharp but starts to look unnatural (classic CFG over-saturation)

---
## Recap

The cells below reproduce the schedule, forward process, and DDIM step from earlier notebooks. They are given — not exercises.

In [ ]:
T = 1000


def cosine_schedule(T: int = 1000, s: float = 0.008) -> dict:
    steps = jnp.arange(T + 1)
    alphas_bar_full = jnp.cos((steps / T + s) / (1 + s) * jnp.pi / 2) ** 2
    alphas_bar_full = alphas_bar_full / alphas_bar_full[0]
    betas = 1 - alphas_bar_full[1:] / alphas_bar_full[:-1]
    betas = jnp.clip(betas, 0, 0.999)
    alphas = 1.0 - betas
    return {"betas": betas, "alphas": alphas, "alphas_bar": alphas_bar_full[1:]}


def q_sample(
    x0: Float[Array, "h w c"],
    t: Int[Array, ""],
    noise: Float[Array, "h w c"],
    alphas_bar: Float[Array, " T"],
) -> Float[Array, "h w c"]:
    ab_t = alphas_bar[t]
    return jnp.sqrt(ab_t) * x0 + jnp.sqrt(1.0 - ab_t) * noise


def sinusoidal_embedding(t: int, dim: int = 256) -> Float[Array, " dim"]:
    half = dim // 2
    freqs = jnp.exp(-jnp.log(10000) * jnp.arange(half, dtype=jnp.float32) / (half - 1))
    args = t * freqs
    return jnp.concatenate([jnp.sin(args), jnp.cos(args)])


def ddim_step(
    xt: Float[Array, "h w c"],
    t_from: int,
    t_to: int,
    eps_pred: Float[Array, "h w c"],
    schedule: dict,
) -> Float[Array, "h w c"]:
    ab_from = schedule["alphas_bar"][t_from]
    ab_to = schedule["alphas_bar"][t_to]
    x_hat0 = (xt - jnp.sqrt(1.0 - ab_from) * eps_pred) / jnp.sqrt(ab_from)
    x_hat0 = jnp.clip(x_hat0, -1.0, 1.0)  # keep the chain in the valid pixel range
    return jnp.sqrt(ab_to) * x_hat0 + jnp.sqrt(1.0 - ab_to) * eps_pred


schedule = cosine_schedule(T)

In [16]:
hf_ds = load_dataset("uoft-cs/cifar10", split="train[:5000]")
images = (
    jnp.array(np.stack([np.array(item["img"]) for item in hf_ds]), dtype=jnp.float32)
    / 127.5
    - 1.0
)
labels = jnp.array([item["label"] for item in hf_ds], dtype=jnp.int32)

print(f"images: {images.shape}   labels: {labels.shape}")
print(
    f"class distribution: { {n: int((labels == i).sum()) for i, n in enumerate(CLASS_NAMES)} }"
)

'[Errno 8] nodename nor servname provided, or not known' thrown while requesting HEAD https://huggingface.co/datasets/uoft-cs/cifar10/resolve/0b2714987fa478483af9968de7c934580d0bb9a2/cifar10.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since uoft-cs/cifar10 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'plain_text' at /Users/maigimenez/.cache/huggingface/datasets/uoft-cs___cifar10/plain_text/0.0.0/0b2714987fa478483af9968de7c934580d0bb9a2 (last modified on Wed Jul  8 03:17:05 2026).


images: (5000, 32, 32, 3)   labels: (5000,)
class distribution: {'airplane': 479, 'automobile': 479, 'bird': 525, 'cat': 471, 'deer': 525, 'dog': 520, 'frog': 509, 'horse': 482, 'ship': 520, 'truck': 490}


---
## 2. Class conditioning

We add a **class embedding table**: a learnable matrix of shape `[11, 256]` — one row per CIFAR-10 class plus one null row (`NULL_CLASS = 10`). During the forward pass, we look up the row for class `c` and **add** it to the projected time embedding before the MLP.

```
t → sinusoidal_embedding → FC(256) → tanh → [256]   ─┐
                                                     ├─ + → [256] → concat(flatten(xt)) → MLP
c → class_emb[c] ────────────────────────────────────┘
```

Because the class embedding has the same dimension as the time embedding, the rest of the architecture — including the weights of `fc1`, `fc2`, `fc3` remains **unchanged**. 
This means we can load the notebook 03 checkpoint and only add the new `class_emb` parameter.

### Exercise 1: extend `init_params` with a class embedding table

Add a `"class_emb"` key to the returned dictionary: a `[11, d_time]` array of small random values.

*Why near-zero?* Starting the class embeddings close to zero means the model initially ignores the class signal — it still denoises correctly using its pre-trained weights — and gradually learns to use it during fine-tuning.

In [ ]:
def init_params(
    key: PRNGKeyArray,
    d_img: int = 3072,
    d_time: int = 256,
    d_hidden: int = 512,
    n_classes: int = 11,  # 10 CIFAR-10 classes + 1 null
) -> dict:
    k0, k1, k2, k3, k4 = jr.split(key, 5)

    def linear(k, fan_in, fan_out):
        return {
            "W": jr.normal(k, (fan_in, fan_out)) * jnp.sqrt(2.0 / fan_in),
            "b": jnp.zeros(fan_out),
        }

    return {
        "time_proj": linear(k0, d_time, d_time),
        "fc1": linear(k1, d_img + d_time, d_hidden),
        "fc2": linear(k2, d_hidden, d_hidden),
        "fc3": linear(k3, d_hidden, d_img),
        # YOUR CODE HERE — add "class_emb" using k4
    }

In [9]:
# @title 💡 Solution (open to reveal after trying to implement)


def init_params(
    key: PRNGKeyArray,
    d_img: int = 3072,
    d_time: int = 256,
    d_hidden: int = 512,
    n_classes: int = 11,
) -> dict:
    k0, k1, k2, k3, k4 = jr.split(key, 5)

    def linear(k, fan_in, fan_out):
        return {
            "W": jr.normal(k, (fan_in, fan_out)) * jnp.sqrt(2.0 / fan_in),
            "b": jnp.zeros(fan_out),
        }

    return {
        "time_proj": linear(k0, d_time, d_time),
        "fc1": linear(k1, d_img + d_time, d_hidden),
        "fc2": linear(k2, d_hidden, d_hidden),
        "fc3": linear(k3, d_hidden, d_img),
        "class_emb": jr.normal(k4, (n_classes, d_time)) * 0.02,
    }

In [10]:
test_params = init_params(jr.PRNGKey(0))
assert "class_emb" in test_params, "class_emb missing from params"
assert test_params["class_emb"].shape == (11, 256), (
    f"expected (11, 256), got {test_params['class_emb'].shape}"
)
print("✓ init_params has class_emb")

✓ init_params has class_emb


### Exercise 2: extend `predict_noise` to accept a class label

Add a `c: int` argument (default `NULL_CLASS = 10`). Inside the forward pass:
1. Look up `params["class_emb"][c]` — shape `[256]`
2. **Add** it to the projected time embedding `t_emb` (element-wise, before concatenation with the flattened image)

Everything else stays the same.

In [ ]:
def predict_noise(
    params: dict,
    xt: Float[Array, "h w c"],
    t: int,
    c: int = NULL_CLASS,
) -> Float[Array, "h w c"]:
    t_emb = sinusoidal_embedding(t)
    t_emb = jnp.tanh(t_emb @ params["time_proj"]["W"] + params["time_proj"]["b"])
    # YOUR CODE HERE — add class embedding to t_emb

    x = jnp.concatenate([xt.reshape(-1), t_emb])
    x = jax.nn.gelu(x @ params["fc1"]["W"] + params["fc1"]["b"])
    x = jax.nn.gelu(x @ params["fc2"]["W"] + params["fc2"]["b"])
    return (x @ params["fc3"]["W"] + params["fc3"]["b"]).reshape(xt.shape)

In [11]:
# @title 💡 Solution (open to reveal after trying to implement)


def predict_noise(
    params: dict,
    xt: Float[Array, "h w c"],
    t: int,
    c: int = NULL_CLASS,
) -> Float[Array, "h w c"]:
    t_emb = sinusoidal_embedding(t)
    t_emb = jnp.tanh(t_emb @ params["time_proj"]["W"] + params["time_proj"]["b"])
    t_emb = t_emb + params["class_emb"][c]  # add class embedding

    x = jnp.concatenate([xt.reshape(-1), t_emb])
    x = jax.nn.gelu(x @ params["fc1"]["W"] + params["fc1"]["b"])
    x = jax.nn.gelu(x @ params["fc2"]["W"] + params["fc2"]["b"])
    return (x @ params["fc3"]["W"] + params["fc3"]["b"]).reshape(xt.shape)

In [12]:
test_params = init_params(jr.PRNGKey(0))
xt_test = jr.normal(jr.PRNGKey(1), (32, 32, 3))

pred_null = predict_noise(test_params, xt_test, t=100, c=NULL_CLASS)
pred_plane = predict_noise(test_params, xt_test, t=100, c=0)
pred_dog = predict_noise(test_params, xt_test, t=100, c=5)

assert pred_null.shape == (32, 32, 3)
assert not jnp.allclose(pred_null, pred_plane), (
    "null and airplane must give different predictions"
)
assert not jnp.allclose(pred_plane, pred_dog), (
    "different classes must give different predictions"
)
print("✓ predict_noise is class-conditional")

✓ predict_noise is class-conditional


---
## 3. Conditional training

The loss is unchanged — still MSE on noise. The only new thing is **conditioning dropout**: for each image in the batch, the class label is randomly replaced with `NULL_CLASS` with probability `p_uncond = 0.1`.

This forces the model to learn both conditional and unconditional denoising from a single set of weights, which is exactly what CFG needs at inference.

### Exercise 3: extend `train_step` with label masking

The existing train step from notebook 03 takes `(params, opt_state, x0_batch, key, schedule)`. Add a `labels_batch` argument and:
1. Sample a boolean mask (shape `[B]`) where each entry is `True` with probability `p_uncond`
2. Replace masked labels with `NULL_CLASS`
3. Pass the masked labels to `predict_noise` inside `loss_fn`

*Hint*: `jnp.where(mask, NULL_CLASS, labels_batch)` replaces entries where `mask` is `True`.

In [ ]:
optimizer = optax.adam(learning_rate=2e-4)


def train_step(
    params: dict,
    opt_state: optax.OptState,
    x0_batch: Float[Array, "b h w c"],
    labels_batch: Int[Array, " b"],
    key: PRNGKeyArray,
    schedule: dict,
    p_uncond: float = 0.1,
) -> tuple[dict, optax.OptState, Float[Array, ""]]:
    B = x0_batch.shape[0]
    key_t, key_noise, key_mask = jr.split(key, 3)

    ts = jr.randint(key_t, (B,), 0, T)
    noise = jr.normal(key_noise, x0_batch.shape)

    # YOUR CODE HERE — sample mask, replace masked labels with NULL_CLASS
    labels_masked = labels_batch  # replace this line

    def loss_fn(params):
        xt = jax.vmap(q_sample, in_axes=(0, 0, 0, None))(
            x0_batch, ts, noise, schedule["alphas_bar"]
        )
        # YOUR CODE HERE — update predict_noise call to pass labels_masked
        eps_pred = jax.vmap(predict_noise, in_axes=(None, 0, 0))(params, xt, ts)
        return jnp.mean((noise - eps_pred) ** 2)

    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, new_opt_state = optimizer.update(grads, opt_state)
    return optax.apply_updates(params, updates), new_opt_state, loss

In [13]:
# @title 💡 Solution (open to reveal after trying to implement)

optimizer = optax.adam(learning_rate=2e-4)


def train_step(
    params: dict,
    opt_state: optax.OptState,
    x0_batch: Float[Array, "b h w c"],
    labels_batch: Int[Array, " b"],
    key: PRNGKeyArray,
    schedule: dict,
    p_uncond: float = 0.1,
    null_class: int = NULL_CLASS,
) -> tuple[dict, optax.OptState, Float[Array, ""]]:
    B = x0_batch.shape[0]
    key_t, key_noise, key_mask = jr.split(key, 3)

    ts = jr.randint(key_t, (B,), 0, T)
    noise = jr.normal(key_noise, x0_batch.shape)

    mask = jr.bernoulli(key_mask, p_uncond, (B,))
    labels_masked = jnp.where(mask, null_class, labels_batch)

    def loss_fn(params):
        xt = jax.vmap(q_sample, in_axes=(0, 0, 0, None))(
            x0_batch, ts, noise, schedule["alphas_bar"]
        )
        eps_pred = jax.vmap(predict_noise, in_axes=(None, 0, 0, 0))(
            params, xt, ts, labels_masked
        )
        return jnp.mean((noise - eps_pred) ** 2)

    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, new_opt_state = optimizer.update(grads, opt_state)
    return optax.apply_updates(params, updates), new_opt_state, loss

---
## Fine-tuning

We load the notebook 03 checkpoint and add the `class_emb` table — initialised near zero so the model starts as a pure unconditional denoiser and gradually learns to use the class signal. 8 epochs of fine-tuning is enough to see class-conditional structure emerge.

In [14]:
if os.path.exists("params_nb03.pkl"):
    with open("params_nb03.pkl", "rb") as f:
        params = pickle.load(f)
    # Add the class embedding table — the rest of the weights are unchanged
    params["class_emb"] = jr.normal(jr.PRNGKey(99), (11, 256)) * 0.02
    print("Loaded NB03 checkpoint + added class_emb  ✓")
else:
    print(
        "params_nb03.pkl not found — initialising from scratch (quality will be lower)."
    )
    params = init_params(jr.PRNGKey(0))

opt_state = optimizer.init(params)

Loaded NB03 checkpoint + added class_emb  ✓


In [15]:
def finetune(params, opt_state, images, labels, key, n_epochs=8, batch_size=128):
    n = len(images)
    history = []

    for epoch in range(1, n_epochs + 1):
        key, shuffle_key = jr.split(key)
        perm = jr.permutation(shuffle_key, n)
        imgs_s = images[perm]
        lbls_s = labels[perm]

        epoch_losses = []
        for i in range(0, n - batch_size + 1, batch_size):
            batch_imgs = imgs_s[i : i + batch_size]
            batch_lbls = lbls_s[i : i + batch_size]
            key, step_key = jr.split(key)
            params, opt_state, loss = train_step_jit(
                params, opt_state, batch_imgs, batch_lbls, step_key, schedule
            )
            epoch_losses.append(float(loss))

        mean_loss = float(np.mean(epoch_losses))
        history.append(mean_loss)
        print(f"epoch {epoch:2d}/{n_epochs}  loss = {mean_loss:.4f}")

    return params, opt_state, history


train_step_jit = jax.jit(train_step)
key = jr.PRNGKey(7)
params, opt_state, history = finetune(params, opt_state, images, labels, key)

NameError: name 'images' is not defined

In [ ]:
with open("params_nb05.pkl", "wb") as f:
    pickle.dump(params, f)
print("Saved → params_nb05.pkl")

In [ ]:
fig = go.Figure(go.Scatter(y=history, mode="lines+markers", name="fine-tune loss"))
fig.update_layout(
    title="Fine-tuning loss", xaxis_title="epoch", yaxis_title="MSE", height=300
)
fig.show()

---
## 4. Guided sampling

The DDIM loop from notebook 04 is unchanged. We only replace the noise estimate: instead of one forward pass, we run two — one with the class, one without — and combine them. Noteworthy, that this willmake our sampling twice more expensive that before.

In [17]:
# Given — the CFG noise estimate. Read it carefully before exercise 4.


def cfg_noise_estimate(
    params: dict,
    xt: Float[Array, "h w c"],
    t: int,
    c: int,
    w: float,
) -> Float[Array, "h w c"]:
    eps_uncond = predict_noise(params, xt, t, c=NULL_CLASS)
    eps_cond = predict_noise(params, xt, t, c=c)
    return eps_uncond + w * (eps_cond - eps_uncond)

### Exercise 4: implement `ddim_sample_cfg`

Copy the DDIM loop from notebook 04 and replace the `predict_noise(params, xt, t_from)` call with `cfg_noise_estimate(params, xt, t_from, c, w)`.

In [ ]:
def ddim_sample_cfg(
    params: dict,
    schedule: dict,
    key: PRNGKeyArray,
    c: int,
    w: float = 3.0,
    shape: tuple = (32, 32, 3),
    steps: int = 50,
) -> Float[Array, "h w c"]:
    # YOUR CODE HERE
    raise NotImplementedError

In [18]:
# @title 💡 Solution (open to reveal after trying to implement)


def ddim_sample_cfg(
    params: dict,
    schedule: dict,
    key: PRNGKeyArray,
    c: int,
    w: float = 3.0,
    shape: tuple = (32, 32, 3),
    steps: int = 50,
) -> Float[Array, "h w c"]:
    timesteps = np.linspace(T - 1, 0, steps + 1).round().astype(int)
    xt = jr.normal(key, shape)
    for i in range(len(timesteps) - 1):
        t_from = int(timesteps[i])
        t_to = int(timesteps[i + 1])
        eps_pred = cfg_noise_estimate(params, xt, t_from, c, w)
        xt = ddim_step(xt, t_from, t_to, eps_pred, schedule)
    return xt

### Class-conditional generation

The grid below generates 4 samples for each of the 10 CIFAR-10 classes at guidance scale `w = 3`. This is the payoff: for the first time you can choose what you generate.

In [19]:
def to_uint8(arr):
    arr = np.array(arr)
    return np.clip((arr + 1) / 2 * 255, 0, 255).astype(np.uint8)


W = 3.0
N_COLS = 4

fig = make_subplots(
    rows=10,
    cols=N_COLS,
    row_titles=CLASS_NAMES,
    horizontal_spacing=0.02,
    vertical_spacing=0.03,
)

key = jr.PRNGKey(42)
for row, class_id in enumerate(range(10), start=1):
    for col in range(1, N_COLS + 1):
        key, sample_key = jr.split(key)
        img = ddim_sample_cfg(params, schedule, sample_key, c=class_id, w=W, steps=50)
        fig.add_trace(go.Image(z=to_uint8(img)), row=row, col=col)

fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)
fig.update_layout(
    title=f"Class-conditional samples  (w = {W}, DDIM 50 steps)",
    height=1000,
)
fig.show()

---
## 5 — Guidance scale sweep

Higher `w` pushes the sample further in the direction of the class making the image sharper and more class-faithful, but less diverse and eventually over-saturated. The sweep below fixes one seed and one class so you can isolate the effect of `w`.

In [21]:
CLASS_ID = 5  # dog
SEED = jr.PRNGKey(17)
W_VALUES = [1.0, 2.0, 4.0, 7.0, 12.0]

fig = make_subplots(
    rows=1,
    cols=len(W_VALUES),
    subplot_titles=[f"w = {w}" for w in W_VALUES],
    horizontal_spacing=0.03,
)
for col, w in enumerate(W_VALUES, start=1):
    img = ddim_sample_cfg(params, schedule, SEED, c=CLASS_ID, w=w, steps=50)
    fig.add_trace(go.Image(z=to_uint8(img)), row=1, col=col)

fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)
fig.update_layout(
    title=f"Guidance scale sweep — class: {CLASS_NAMES[CLASS_ID]}  (same seed across all)",
    height=220,
)
fig.show()

---
## What's next?

In **06. Full DDPM** you will swap the MLP denoiser for a **Diffusion Transformer (DiT)**: patch embeddings, self-attention, and Adaptive LayerNorm (AdaLN) conditioning on both time and class. The CFG sampler from this notebook (`ddim_sample_cfg`) plugs in unchanged — only `predict_noise` is replaced.

The DiT processes CIFAR-10 images as a sequence of $8 \times 8 = 64$ patches, letting attention capture long-range spatial structure that the MLP cannot. You should expect a visible jump in generation quality.